# 9.6 · 损失函数 / Loss Functions

> **课程定位 / Where this fits**
> 第 6 课，**Part 9 · 深度学习基础**。
> Lesson 6, **Part 9 · Deep Learning Foundations**.
>
> 损失函数定义了网络**到底在优化什么**——选错损失，模型再大也学不到你想要的东西。这一课系统过一遍深度学习常用损失：回归的 **MSE/MAE/Huber**、分类的**交叉熵**、不平衡的 **Focal Loss**、以及度量学习的**对比/三元组损失**。重点是理解每个损失**为什么这样设计、配什么任务**。
> The loss defines **what the network actually optimizes** — pick the wrong one and even a huge model learns the wrong thing. This lesson surveys common DL losses: regression **MSE/MAE/Huber**, classification **cross-entropy**, imbalance **focal loss**, and metric-learning **contrastive/triplet** losses. The focus is *why* each is designed that way and *which task* it fits.
>
> 💼 **实战/面试视角**："为什么分类用交叉熵不用 MSE / Focal Loss 解决什么 / 三元组损失干什么" 高频。
> 💼 **Practical/interview angle:** "why cross-entropy not MSE for classification / what focal loss fixes / what triplet loss does" — common.

> 📐 **符号约定 / Notation**
> - $y, \hat y$ —— 真实值、预测值 / target, prediction
> - $p$ —— 预测概率 / predicted probability

> 💡 **面试相关 / Interview-relevant**
> - "分类为什么用交叉熵不用 MSE"（出镜率 ★★★★★）
> - "Focal Loss 怎么处理类别不平衡"（★★★★）
> - "Huber loss vs MSE/MAE"（★★★★）
> - "对比损失/三元组损失干什么（度量学习）"（★★★★）
> - "损失函数怎么对应任务"（★★★★）

---

## 学习目标 / Learning Objectives

1. 回顾回归损失 MSE/MAE/**Huber**。
   Review regression losses MSE/MAE/Huber.
2. 复盘分类的**交叉熵**、为什么不用 MSE。
   Recap classification cross-entropy and why not MSE.
3. 理解 **Focal Loss** 怎么聚焦难样本、治不平衡。
   Understand how Focal Loss focuses hard samples and fixes imbalance.
4. 理解度量学习的**对比/三元组损失**。
   Understand contrastive/triplet losses for metric learning.
5. 学会**按任务选损失**。
   Learn to pick the loss by task.

## 目录 / TOC
1. [先建直觉：损失 = 优化目标 ⭐](#1)
2. [回归损失：MSE/MAE/Huber ⭐](#2)
3. [分类：交叉熵 vs MSE ⭐](#3)
4. [Focal Loss：聚焦难样本 ⭐](#4)
5. [度量学习：对比/三元组 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：损失 = 优化目标 ⭐ / Intuition: Loss = the Objective

神经网络训练就是"调权重让损失最小"。所以**损失函数就是你给模型下的'目标定义'**——它精确规定了"什么样的预测算好、什么算差、差多少"。同一个网络，配不同损失会学出完全不同的行为。
Training a network means "adjust weights to minimize the loss". So the **loss function is your "definition of the goal"** — it precisely specifies "what counts as a good/bad prediction and how bad". The same network with different losses learns completely different behavior.

选损失的两条主线：(1) **任务类型**——回归 vs 分类 vs 度量学习，决定损失大类；(2) **特殊需求**——异常值多？类别不平衡？要学相似度？决定具体选哪个。本课就沿这两条线走。
Two threads for choosing a loss: (1) **task type** — regression vs classification vs metric learning, decides the family; (2) **special needs** — many outliers? class imbalance? learning similarity? decides the specific choice. We follow both.


<a id="2"></a>
## 2. 回归损失：MSE/MAE/Huber ⭐ / Regression Losses

回归预测连续值（7.1 从指标角度讲过，这里从**损失/训练**角度）：
Regression predicts continuous values (7.1 covered the metrics angle; here the **loss/training** angle):
- **MSE（平方误差）**：$(y-\hat y)^2$。处处可导、对应高斯噪声 MLE，但**对异常值极敏感**（误差平方放大）。
  **MSE:** $(y-\hat y)^2$. Smooth, the Gaussian-noise MLE, but **very outlier-sensitive** (squaring amplifies).
- **MAE（绝对误差）**：$|y-\hat y|$。**对异常值稳健**（线性惩罚），但在 0 处不可导、梯度不随误差大小变化（优化稍难）。
  **MAE:** $|y-\hat y|$. **Robust to outliers** (linear penalty), but non-differentiable at 0 and its gradient doesn't scale with error.
- **Huber**：小误差用平方（光滑、好优化）、大误差用线性（抗异常）——**两者的最佳折中**（4.15 详讲）。
  **Huber:** squared for small errors (smooth) and linear for large (robust) — **the best of both** (detailed in 4.15).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
sns.set_theme(style="whitegrid")

err = np.linspace(-3, 3, 300)                              # 误差 = y - ŷ
delta = 1.0
mse = err**2
mae = np.abs(err)
huber = np.where(np.abs(err) <= delta, 0.5*err**2, delta*(np.abs(err)-0.5*delta))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(err, mse, label="MSE (平方, 对异常敏感)", lw=2)
ax.plot(err, mae, label="MAE (绝对, 抗异常)", lw=2)
ax.plot(err, huber, label=f"Huber (δ={delta}, 折中)", lw=2)
ax.set_xlabel("误差 error (y-ŷ)"); ax.set_ylabel("loss"); ax.legend()
ax.set_title("回归损失: MSE 对大误差惩罚平方增长(敏感); MAE 线性(稳健); Huber 两者折中")
plt.tight_layout(); plt.show()
print("误差大时: MSE 平方爆炸(被异常值主导); MAE/Huber 线性(稳健)")
print("选择: 无异常→MSE; 有异常→MAE/Huber; Huber 兼顾光滑与稳健(回归默认好选择)")


<a id="3"></a>
## 3. 分类：交叉熵 vs MSE ⭐ / Classification: Cross-Entropy vs MSE

分类**永远用交叉熵**，不用 MSE（5.1 推导过，这里从损失曲线角度再强化，因为是绝对高频题）。两个原因：
Classification **always uses cross-entropy**, not MSE (derived in 5.1; reinforced here from the loss-curve angle since it's a top question). Two reasons:
- **凸性**：交叉熵关于网络输出是凸的（配 softmax），MSE+sigmoid 非凸 → 易陷局部最优。
  **Convexity:** cross-entropy is convex w.r.t. outputs (with softmax); MSE+sigmoid is non-convex → local optima.
- **梯度不消失**：交叉熵在"自信地错"时梯度**很大**（罚得狠、学得快），MSE+sigmoid 此时梯度**趋于 0**（学不动）——这才是关键。
  **Non-vanishing gradient:** when "confidently wrong", cross-entropy's gradient is **large** (penalizes hard, learns fast); MSE+sigmoid's gradient **goes to 0** (can't learn) — the key reason.

下面画"真实标签=1 时，预测概率从 1 到 0"的损失曲线对比：交叉熵在 p→0（自信地错）时**飙到无穷**，MSE 只到 1。
We plot the loss as the predicted probability goes from 1 to 0 (truth = 1): cross-entropy **shoots to infinity** as p→0 (confidently wrong), while MSE only reaches 1.


In [ ]:
p = np.linspace(0.001, 0.999, 300)                        # 预测正类的概率(真实=1)
ce_loss = -np.log(p)                                      # 交叉熵: 真实=1 时为 -log(p)
mse_loss = (1 - p)**2                                     # MSE: (1-p)²
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, ce_loss, label="交叉熵 -log(p)", lw=2)
ax.plot(p, mse_loss, label="MSE (1-p)²", lw=2)
ax.set_xlabel("预测概率 p (真实标签=1)"); ax.set_ylabel("loss"); ax.legend(); ax.set_ylim(0, 5)
ax.set_title("真实=1 时: 交叉熵在'自信地错'(p→0)处飙到∞(罚狠学快); MSE 只到 1(梯度趋0学不动)")
plt.tight_layout(); plt.show()
print("交叉熵: 越自信地错, 损失越爆炸 → 梯度大 → 快速纠正; MSE+sigmoid 此时梯度趋0 → 学不动")
print("→ 分类永远用交叉熵(5.1); PyTorch: nn.CrossEntropyLoss(自带 softmax) / BCEWithLogitsLoss(二分类)")


<a id="4"></a>
## 4. Focal Loss：聚焦难样本 ⭐ / Focal Loss: Focus on Hard Samples

**Focal Loss** 是为**极端类别不平衡**（如目标检测里背景占 99.9%）设计的交叉熵改进。问题：海量"容易的"样本（已经分得很对）虽然每个损失小，但**数量太多，加起来淹没了少数难样本**。
**Focal Loss** improves cross-entropy for **extreme class imbalance** (e.g. 99.9% background in object detection). The problem: a flood of "easy" (already-correct) samples, each with small loss, **collectively drowns out the few hard ones**.

Focal Loss 的做法：在交叉熵前乘一个**调制因子** $(1-p_t)^\gamma$——$p_t$ 是模型对正确类的预测概率。已经分对的样本（$p_t$ 大）被乘以一个**很小**的因子，损失贡献被压低；难样本（$p_t$ 小）几乎不受影响。于是训练**自动聚焦在难样本上**。
Focal Loss multiplies cross-entropy by a **modulating factor** $(1-p_t)^\gamma$ — $p_t$ is the predicted probability of the correct class. Easy samples (large $p_t$) get a **tiny** factor, suppressing their contribution; hard samples (small $p_t$) are barely affected. Training **auto-focuses on hard samples**.


In [ ]:
pt = np.linspace(0.001, 0.999, 300)                       # 对正确类的预测概率
ce = -np.log(pt)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(pt, ce, label="交叉熵 (γ=0)", lw=2)
for gamma in [0.5, 2, 5]:
    focal = (1 - pt)**gamma * (-np.log(pt))               # focal = (1-pt)^γ · CE
    ax.plot(pt, focal, label=f"Focal (γ={gamma})", lw=2)
ax.axvspan(0.7, 1.0, alpha=0.08, color="green"); ax.text(0.72, 2, "已分对的'易'样本\n(损失被压低)", fontsize=8)
ax.set_xlabel("对正确类的预测概率 pt"); ax.set_ylabel("loss"); ax.legend()
ax.set_title("Focal Loss: (1-pt)^γ 把'已分对(pt大)'样本的损失压下去 → 聚焦难样本")
plt.tight_layout(); plt.show()
print("γ 越大, 对'已分对的易样本'压得越狠 → 训练自动聚焦在难/少数样本上")
print("用于极端不平衡(目标检测背景占比超高); γ=2 是常用默认; 也可加类别权重 α")


<a id="5"></a>
## 5. 度量学习：对比/三元组 + 小结 ⭐ / Metric Learning & Summary

前面的损失都在"预测一个标签/数值"。**度量学习(metric learning)** 换了个目标：**学一个嵌入空间，让相似的样本靠近、不相似的远离**。用于人脸识别、以图搜图、推荐——这些任务的类别太多/会变，没法直接分类，只能学"相似度"。
The losses above predict "a label/value". **Metric learning** has a different goal: **learn an embedding space where similar samples are close and dissimilar ones far apart**. Used in face recognition, image retrieval, recommendation — tasks with too many/changing classes to classify directly, so we learn "similarity".

- **对比损失(contrastive)**：成对样本。同类对→拉近距离；异类对→推开到至少一个间隔 margin 之外。
  **Contrastive:** on pairs. Same-class pair → pull close; different-class pair → push apart beyond a margin.
- **三元组损失(triplet)**：三元组 (锚点 a, 正例 p, 负例 n)。要求 **a 到 p 的距离 + margin < a 到 n 的距离**——即"正例必须比负例离锚点更近至少一个 margin"。FaceNet 用它做人脸识别。
  **Triplet:** on triplets (anchor a, positive p, negative n). Require **dist(a,p) + margin < dist(a,n)** — the positive must be closer to the anchor than the negative by at least a margin. FaceNet uses it for face recognition.


In [ ]:
# 三元组损失演示: max(0, d(a,p) - d(a,n) + margin) / triplet loss
def triplet_loss(a, p, n, margin=1.0):
    d_ap = ((a - p)**2).sum(-1)**0.5                      # 锚点到正例的距离
    d_an = ((a - n)**2).sum(-1)**0.5                      # 锚点到负例的距离
    return np.maximum(0, d_ap - d_an + margin)           # 要求 d_ap + margin < d_an

a = np.array([0.0, 0.0])
print("三元组损失 = max(0, d(锚,正) - d(锚,负) + margin), 目标: 正例比负例离锚点更近一个 margin")
print(f"\n{'情形':<34}{'d(a,p)':>8}{'d(a,n)':>8}{'损失':>8}")
for desc, p, n in [("好(正近负远)", [0.2,0], [3.0,0]), ("差(正远负近)", [2.0,0], [0.5,0]),
                   ("边界(正负等距)", [1.0,0], [1.0,0])]:
    p, n = np.array(p), np.array(n)
    print(f"{desc:<36}{np.linalg.norm(a-p):>8.2f}{np.linalg.norm(a-n):>8.2f}{triplet_loss(a,p,n):>8.2f}")
print("\n好情形(正例近)损失=0(无需优化); 差情形损失大(推动拉近正例/推远负例)")
print("度量学习用于人脸识别/以图搜图/推荐(类别太多无法直接分类, 改学相似度嵌入)")


```
损失=优化目标; 选损失看 ①任务类型 ②特殊需求(异常/不平衡/相似度)
回归: MSE(光滑, 对异常敏感) / MAE(稳健, 不光滑) / Huber(折中, 默认好选)
分类: 永远交叉熵(凸+梯度不消失); 不用 MSE(非凸+自信错时梯度趋0); 二分类 BCEWithLogits
Focal Loss: (1-pt)^γ·CE, 压低易样本损失 → 聚焦难/少数样本, 治极端不平衡(检测)
度量学习: 学相似度嵌入; 对比损失(成对) / 三元组损失(锚-正-负, 正比负近一个 margin)
PyTorch: nn.MSELoss / HuberLoss / CrossEntropyLoss / BCEWithLogitsLoss / TripletMarginLoss
```

### 💡 面试速查 / Interview cheat-sheet
1. **分类用交叉熵不用 MSE**: 凸 + 自信错时梯度大(MSE 此时趋0)。
   Classification uses cross-entropy not MSE: convex + large gradient when confidently wrong.
2. **回归 Huber** 兼顾 MSE 光滑与 MAE 抗异常。
   Huber combines MSE's smoothness with MAE's robustness.
3. **Focal Loss = (1-pt)^γ·CE**, 压低易样本 → 聚焦难样本(极端不平衡)。
   Focal loss = (1-pt)^γ·CE, suppresses easy samples → focuses on hard ones (extreme imbalance).
4. **度量学习**: 对比/三元组损失学相似度嵌入(人脸/检索)。
   Metric learning: contrastive/triplet losses learn similarity embeddings (face/retrieval).
5. **损失对应任务**: 回归/分类/不平衡/相似度各有对应损失。
   Match loss to task: regression/classification/imbalance/similarity each have their loss.

### 下一节 / Next
**9.7 优化器**——有了损失, 用什么算法最小化它? SGD、Momentum、RMSProp、Adam、AdamW 的演进与对比。
**9.7 Optimizers** — given a loss, what algorithm minimizes it? The evolution and comparison of SGD, Momentum, RMSProp, Adam, AdamW.
